# Manipulating GSIM files - Olha Kachalova

Assignment: manipulate GSIM US*.mon files in order to create n YYYY-MM.txt files, one for each year-month (from the first date to the last date), thet includes the station ID, latitude, longitude, and the MEAN value.

Proposed solution:
    1. Create a single all.txt file, that contains the information needed from all US*.mon files, merged together.
    2. Split all.txt into separate files by unique month-year, removing the first column "date" from the outputs

In [ ]:
! cd /media/sf_LVM_shared/GSIM_indices/TIMESERIES/monthly
! mkdir -p out
! ALL="out/all.txt" #navigate to the folder and create the output folder with empty all.txt

In [ ]:
! printf "date,STATION_ID,latitude,longitude,MEAN\n" > "$ALL" #make header in all.txt

In [ ]:
# create all.csv with the header as requested
%%bash
for f in US*.mon; do
# set variables and grep
    sid="$(grep gsim.no "$f" | awk '{print $4}')"
    lat="$(grep latitude "$f" | awk '{print $4}')"
    lon="$(grep longitude "$f" | awk '{print $4}')"
#transfer bash variables to awk and start with the first row where 
#the first column (date) starts from number = beginning of the dataset
    awk -v sid="$sid" -v lat="$lat" -v lon="$lon" '  
      $1 ~ /^[0-9]/ { print $1, sid, lat, lon, $2 }' "$f" >> "$ALL"
done    

In [ ]:
# remove the date from the header | split by commas and keep the 1st field | take characters 1-7 |
# sort unique month-years, create outputs out/YYYY-MM.txt
%%bash
header=$(head -n1 out/all.txt | cut -d',' -f2-') 
tail -n +2 out/all.txt | cut -d',' -f1 | cut -c1-7 | sort -u | 
while read -r month; do
  { echo "$header"; grep "^$month" out/all.txt | cut -d',' -f2-; } > "out/$month.txt";
done